## Q1 自检

In [ ]:
print("数据质量检查:")
print(f"1. 行数: {len(df1)}")
print(f"2. 列数: {len(df1.columns)}")
print(f"3. 列名: {list(df1.columns)}")
print(f"4. 数据类型:\n{df1.dtypes}")
print(f"5. 缺失值数量:\n{df1.isna().sum()}")
print(f"6. 价格列统计:\n{df1['Price'].describe()}")
print(f"7. 日期范围: {df1['PriceUpdatedDate'].min()} 到 {df1['PriceUpdatedDate'].max()}")
    
# 检查前32000行和后面部分是否连接正确
print("\n连接检查:")
print(f"第32000行: {df1.iloc[31999].to_dict()}")
print(f"第32001行: {df1.iloc[32000].to_dict()}")

## Q2 自检
### 只要满足(59256, 8)的格式基本就没问题

## Q3 自检

In [ ]:
print("===== 问题3自检 =====")

# 检查DataFrame是否存在
if df3 is None:
    print("❌ 失败：DataFrame为空")

# 检查形状
print(f"DataFrame形状: {df3.shape}")
shape_correct = df3.shape == (16875, 6)
print(f"{'✅' if shape_correct else '❌'} 形状检查: {'正确' if shape_correct else '错误，应为(16875, 6)'}")

# 检查列名
expected_columns = ['postcode', 'place_name', 'state_name', 'state_code', 'latitude', 'longitude']
columns_correct = set(df3.columns) == set(expected_columns)
print(f"{'✅' if columns_correct else '❌'} 列名检查: {'正确' if columns_correct else '错误'}")
print(f"实际列名: {list(df3.columns)}")
print(f"期望列名: {expected_columns}")

# 检查accuracy列是否被删除
accuracy_removed = 'accuracy' not in df3.columns
print(f"{'✅' if accuracy_removed else '❌'} accuracy列删除检查: {'已删除' if accuracy_removed else '未删除'}")

# 抽样检查数据
print("\n数据抽样检查 (前5行):")
print(df3.head(10))

# 检查数据类型
print("\n数据类型检查:")
print(df3.dtypes)

# 检查是否有缺失值
missing_values = df3.isnull().sum().sum()
print(f"\n缺失值总数: {missing_values}")

# 计算得分
score = 0
if shape_correct and columns_correct and accuracy_removed:
    score += 0.5  # 数据正确

if accuracy_removed and columns_correct:
    score += 0.5  # 表头和accuracy列删除正确

print(f"\n估计得分: {score}/1.0")
print(f"通过状态: {'通过' if score == 1.0 else '未通过'}")

## Q4 自检

In [ ]:
"""
检查匹配逻辑是否正确实现
"""
print("\n===== 匹配逻辑检查 =====")

# 准备数据
df3_copy = df3.copy()
df3_copy['place_name_upper'] = df3_copy['place_name'].str.upper()

# 结果统计
results = {
    'full_match_correct': 0,
    'full_match_incorrect': 0,
    'postcode_only_match_correct': 0,
    'postcode_only_match_incorrect': 0,
    'no_match_correct': 0,
    'no_match_incorrect': 0
}

# 随机抽样
import random
sample_size = 15
sample_rows = df4.sample(sample_size)

print(f"随机抽样 {sample_size} 行进行检查:")

# 逐行检查
for i, (_, row) in enumerate(sample_rows.iterrows()):
    postcode = row['Postcode']
    suburb = row['Suburb']
    lat = row['Latitude']
    long = row['Longitude']

    print(f"\n样本 {i+1}: Postcode={postcode}, Suburb={suburb}")

    # 1. 检查是否有完全匹配
    full_matches = df3_copy[(df3_copy['postcode'] == postcode) & 
                           (df3_copy['place_name_upper'] == suburb)]

    # 2. 检查是否有邮编匹配
    postcode_matches = df3_copy[df3_copy['postcode'] == postcode].sort_values('place_name')

    # 3. 确定预期值
    if len(full_matches) > 0:
        # 应该使用完全匹配
        match_type = "完全匹配"
        expected_lat = full_matches.iloc[0]['latitude']
        expected_long = full_matches.iloc[0]['longitude']
    elif len(postcode_matches) > 0:
        # 应该使用字母排序第一的邮编匹配
        match_type = "仅邮编匹配"
        expected_lat = postcode_matches.iloc[0]['latitude']
        expected_long = postcode_matches.iloc[0]['longitude']
        print(f"  可能的匹配: {', '.join(postcode_matches['place_name'].tolist())}")
        print(f"  按字母排序第一个: {postcode_matches.iloc[0]['place_name']}")
    else:
        # 应该为空
        match_type = "无匹配"
        expected_lat = None
        expected_long = None

    # 4. 检查实际值是否符合预期
    if match_type == "完全匹配":
        is_correct = (lat == expected_lat and long == expected_long)
        if is_correct:
            results['full_match_correct'] += 1
        else:
            results['full_match_incorrect'] += 1
        print(f"  {'✅' if is_correct else '❌'} {match_type}: 预期({expected_lat}, {expected_long}), 实际({lat}, {long})")

    elif match_type == "仅邮编匹配":
        is_correct = (lat == expected_lat and long == expected_long)
        if is_correct:
            results['postcode_only_match_correct'] += 1
        else:
            results['postcode_only_match_incorrect'] += 1
        print(f"  {'✅' if is_correct else '❌'} {match_type}: 预期({expected_lat}, {expected_long}), 实际({lat}, {long})")

    else:  # 无匹配
        is_correct = (pd.isna(lat) and pd.isna(long))
        if is_correct:
            results['no_match_correct'] += 1
        else:
            results['no_match_incorrect'] += 1
        print(f"  {'✅' if is_correct else '❌'} {match_type}: 预期(空值), 实际({lat}, {long})")

# 统计结果
print("\n===== 检查结果统计 =====")
print(f"完全匹配: {results['full_match_correct']}正确, {results['full_match_incorrect']}错误")
print(f"仅邮编匹配: {results['postcode_only_match_correct']}正确, {results['postcode_only_match_incorrect']}错误")
print(f"无匹配: {results['no_match_correct']}正确, {results['no_match_incorrect']}错误")

# 特别检查仅邮编匹配的情况
if results['postcode_only_match_correct'] + results['postcode_only_match_incorrect'] == 0:
    print("\n⚠️ 警告: 抽样中没有发现仅邮编匹配的情况")
    print("进行额外检查...")

    # 找出有多个地名的邮编
    postcode_counts = df3_copy['postcode'].value_counts()
    multiple_match_postcodes = postcode_counts[postcode_counts > 1].index.tolist()

    if multiple_match_postcodes:
        # 随机选择5个有多个匹配的邮编
        sample_postcodes = random.sample(multiple_match_postcodes, min(5, len(multiple_match_postcodes)))

        # 创建测试案例
        test_cases = []
        for postcode in sample_postcodes:
            matches = df3_copy[df3_copy['postcode'] == postcode]
            # 创建一个不存在的suburb名称
            fake_suburb = "NONEXISTENT_" + str(random.randint(1000, 9999))
            test_cases.append((postcode, fake_suburb))

        print(f"创建 {len(test_cases)} 个测试案例:")

        for postcode, fake_suburb in test_cases:
            print(f"\n测试案例: Postcode={postcode}, Suburb={fake_suburb} (不存在)")

            # 在df4中查找使用此邮编的行
            matching_rows = df4[df4['Postcode'] == postcode].head(2)

            if len(matching_rows) > 0:
                for _, row in matching_rows.iterrows():
                    print(f"  找到使用此邮编的行: Suburb={row['Suburb']}")
                    print(f"  Latitude={row['Latitude']}, Longitude={row['Longitude']}")

                    # 检查是否使用了字母排序第一的地名
                    sorted_matches = df3_copy[df3_copy['postcode'] == postcode].sort_values('place_name')
                    expected_lat = sorted_matches.iloc[0]['latitude']
                    expected_long = sorted_matches.iloc[0]['longitude']

                    print(f"  按字母排序第一个地名: {sorted_matches.iloc[0]['place_name']}")
                    print(f"  预期经纬度: ({expected_lat}, {expected_long})")

                    # 如果这行的Suburb不是完全匹配，检查是否使用了字母排序第一的地名
                    if row['Suburb'] != sorted_matches.iloc[0]['place_name_upper']:
                        is_correct = (row['Latitude'] == expected_lat and row['Longitude'] == expected_long)
                        if is_correct:
                            results['postcode_only_match_correct'] += 1
                        else:
                            results['postcode_only_match_incorrect'] += 1
                        print(f"  {'✅' if is_correct else '❌'} 仅邮编匹配: 预期({expected_lat}, {expected_long}), 实际({row['Latitude']}, {row['Longitude']})")
            else:
                print("  没有找到使用此邮编的行")

# 最终结论
print("\n===== 最终结论 =====")

# 计算成功率
total_checks = sum(results.values())
total_correct = results['full_match_correct'] + results['postcode_only_match_correct'] + results['no_match_correct']
success_rate = total_correct / total_checks if total_checks > 0 else 0

# 检查是否有仅邮编匹配的情况
has_postcode_only_match = (results['postcode_only_match_correct'] + results['postcode_only_match_incorrect']) > 0

# 最终评估
if success_rate >= 0.95:  # 95%以上正确率
    print(f"✅ 通过! 成功率: {success_rate*100:.1f}%")
    if not has_postcode_only_match:
        print("⚠️ 注意: 未发现仅邮编匹配的情况，但其他匹配逻辑正确")
else:
    print(f"❌ 未通过! 成功率: {success_rate*100:.1f}%")

    # 指出具体问题
    if results['full_match_incorrect'] > 0:
        print(f"- 完全匹配错误: {results['full_match_incorrect']}个")
    if results['postcode_only_match_incorrect'] > 0:
        print(f"- 仅邮编匹配错误: {results['postcode_only_match_incorrect']}个")
    if results['no_match_incorrect'] > 0:
        print(f"- 无匹配错误: {results['no_match_incorrect']}个")
    if not has_postcode_only_match:
        print("- 未发现仅邮编匹配的情况，无法验证字母排序逻辑")

## Q5自检